# 04 - Regression-through-Classification ResNet-50 K-Fold

Trains a ResNet-50 model that treats ages 1 to 100 as classes and evaluates predictions using MAE.

**Dataset:** UTKFace face images. The expected filename format is `age_gender_race_date.jpg`, or images can be organized into age folders using the preprocessing scripts.

**Validation:** 5-fold cross-validation. The same fold structure is reused across the base models and fusion model so the comparisons remain consistent.

**Output:** Fold-specific checkpoints are saved under `checkpoints/classification_resnet_kfold/`.

**Run order:** notebooks 01 and 02 must be run before notebook 03. Notebooks 04 and 05 must be run before notebook 06.


In [ ]:
# Regression through Classification - ResNet50 - KFold
# folds are read from UTKFace_folds/fold_0 ... fold_4

import os
import glob
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from torchvision import transforms
try:
    from torchvision.models import resnet50, ResNet50_Weights
    _HAS_NEW_TORCHVISION_WEIGHTS = True
except Exception:
    from torchvision.models import resnet50
    ResNet50_Weights = None
    _HAS_NEW_TORCHVISION_WEIGHTS = False

from transformers import DeiTForImageClassification
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import mean_absolute_error, accuracy_score
from tqdm.auto import tqdm

In [ ]:
# Settings
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This keeps paths working whether the notebook is run from the project root
# or from inside the notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Raw UTKFace folder used only as fallback if prepared folds are not found.
DATA_ROOT = PROJECT_ROOT / "UTKFace"

# Folders created by preprocessing/split_folds.py
FOLDS_ROOT = PROJECT_ROOT / "UTKFace_folds"
USE_PREMADE_FOLDS_IF_AVAILABLE = True

K_FOLDS = 5
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

EPOCHS = 200
PATIENCE = 7
WEIGHT_DECAY = 0.01
NUM_CLASSES = 100

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)


def extract_age_from_filename(path):
    name = Path(path).name
    try:
        age = int(name.split("_")[0])
    except Exception as exc:
        raise ValueError(f"Could not extract age from filename: {name}") from exc
    return int(min(max(age, 1), NUM_CLASSES))


def list_image_files(root: Path):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    files = sorted([p for p in files if p.is_file()])
    return files


def load_paths_and_ages(root: Path):
    image_paths = list_image_files(root)
    cleaned_paths, ages = [], []
    for p in image_paths:
        try:
            age = extract_age_from_filename(p)
            cleaned_paths.append(str(p))
            ages.append(age)
        except ValueError:
            # skip non-image or badly named files
            continue
    if len(cleaned_paths) == 0:
        raise FileNotFoundError(
            f"No UTKFace-style images found under {root}. "
            "Set DATA_ROOT or FOLDS_ROOT correctly."
        )
    return np.array(cleaned_paths), np.array(ages)


def load_premade_fold_paths(folds_root: Path):
    fold_dirs = sorted([p for p in folds_root.glob("fold_*") if p.is_dir()])
    folds = []
    for fold_dir in fold_dirs:
        train_dir = fold_dir / "train"
        test_dir = fold_dir / "test"
        if not train_dir.exists() or not test_dir.exists():
            continue

        train_paths, train_ages = load_paths_and_ages(train_dir)
        val_paths, val_ages = load_paths_and_ages(test_dir)
        folds.append((train_paths, train_ages, val_paths, val_ages))

    return folds


def make_dynamic_kfolds(data_root: Path):
    image_paths, ages = load_paths_and_ages(data_root)

    # keep the age distribution close across folds
    age_bins = np.array([min((int(a) - 1) // 10, 9) for a in ages])

    try:
        splitter = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
        split_iter = splitter.split(image_paths, age_bins)
        folds = [
            (image_paths[tr], ages[tr], image_paths[va], ages[va])
            for tr, va in split_iter
        ]
    except ValueError:
        # use plain k-fold if stratification fails
        splitter = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
        split_iter = splitter.split(image_paths)
        folds = [
            (image_paths[tr], ages[tr], image_paths[va], ages[va])
            for tr, va in split_iter
        ]

    return folds


def get_folds():
    if USE_PREMADE_FOLDS_IF_AVAILABLE and FOLDS_ROOT.exists():
        folds = load_premade_fold_paths(FOLDS_ROOT)
        if len(folds) != K_FOLDS:
            raise ValueError(f"Expected {K_FOLDS} folds in {FOLDS_ROOT}, found {len(folds)}.")
        print(f"Using {len(folds)} folds from: {FOLDS_ROOT}")
        return folds

    print(f"Using dynamic {K_FOLDS}-fold split from: {DATA_ROOT}")
    return make_dynamic_kfolds(DATA_ROOT)


class UTKFaceDataset(Dataset):
    def __init__(self, image_paths, ages, transform=None, task="regression"):
        self.image_paths = list(image_paths)
        self.ages = [int(a) for a in ages]
        self.transform = transform
        self.task = task

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        age = int(min(max(self.ages[idx], 1), NUM_CLASSES))

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.task == "classification":
            # class 0 means age 1
            target = torch.tensor(age - 1, dtype=torch.long)
        else:
            target = torch.tensor(float(age), dtype=torch.float32)

        return image, target


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])


def make_loaders(train_paths, train_ages, val_paths, val_ages, task):
    train_ds = UTKFaceDataset(train_paths, train_ages, transform=train_transform, task=task)
    val_ds = UTKFaceDataset(val_paths, val_ages, transform=val_transform, task=task)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader

In [ ]:
def build_resnet50_backbone():
    if _HAS_NEW_TORCHVISION_WEIGHTS:
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    else:
        model = resnet50(pretrained=True)
    return model


def set_resnet_trainable_layers(resnet_model, last_conv_layers: int):
    # last 39: layer2-layer4, last 48: layer1-layer4
    for p in resnet_model.parameters():
        p.requires_grad = False

    if last_conv_layers >= 48:
        groups_to_unfreeze = ["layer1", "layer2", "layer3", "layer4"]
    elif last_conv_layers >= 39:
        groups_to_unfreeze = ["layer2", "layer3", "layer4"]
    elif last_conv_layers >= 27:
        groups_to_unfreeze = ["layer3", "layer4"]
    else:
        groups_to_unfreeze = ["layer4"]

    for group_name in groups_to_unfreeze:
        for p in getattr(resnet_model, group_name).parameters():
            p.requires_grad = True

In [ ]:
# Best ResNet-50 regression-through-classification setup from the thesis table:
# - unfreeze the last 48 convolutional layers
# - dropout: 0.3
# - prediction head: 2048 -> 1024 -> 512 -> 100 age classes
RUN_NAME = "classification_resnet_kfold"
LR = 1e-5
RESNET_LAST_CONV_LAYERS = 48


class ResNetAgeClassifier(nn.Module):
    def __init__(self, num_classes=100, last_conv_layers=48, dropout=0.3):
        super().__init__()

        self.resnet = build_resnet50_backbone()
        cnn_dim = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        set_resnet_trainable_layers(self.resnet, last_conv_layers)

        self.feature_processor = nn.Sequential(
            nn.Linear(cnn_dim, 1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(512, num_classes)

    def forward_features(self, x):
        cnn_out = self.resnet(x)
        return self.feature_processor(cnn_out)

    def forward(self, x):
        features = self.forward_features(x)
        return self.classifier(features)


def build_model(fold):
    return ResNetAgeClassifier(
        num_classes=NUM_CLASSES,
        last_conv_layers=RESNET_LAST_CONV_LAYERS,
        dropout=0.3,
    )

In [ ]:
def train_one_epoch_classification(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


@torch.no_grad()
def evaluate_classification_as_regression(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_true, all_classes = [], [], []

    age_values = torch.arange(1, NUM_CLASSES + 1, dtype=torch.float32, device=device)

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item()

        probs = torch.softmax(logits, dim=1)
        pred_ages = torch.sum(probs * age_values.unsqueeze(0), dim=1)
        pred_ages = torch.round(pred_ages).clamp(1, NUM_CLASSES)

        pred_classes = torch.argmax(logits, dim=1)

        all_preds.extend(pred_ages.detach().cpu().numpy())
        all_true.extend((labels + 1).detach().cpu().numpy())
        all_classes.extend(pred_classes.detach().cpu().numpy())

    mae = mean_absolute_error(all_true, all_preds)
    acc = accuracy_score(np.array(all_true) - 1, np.array(all_classes))

    return {
        "loss": total_loss / max(len(loader), 1),
        "mae": mae,
        "accuracy": acc,
    }


def save_checkpoint(path, model, optimizer, fold, epoch, metrics, extra_config=None):
    payload = {
        "fold": fold,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "extra_config": extra_config or {},
    }
    torch.save(payload, path)


def run_classification_kfold(model_builder, run_name, lr, optimize_fusion_head_only=False):
    folds = get_folds()
    all_best_rows = []
    run_ckpt_dir = CHECKPOINT_DIR / run_name
    run_out_dir = OUTPUT_DIR / run_name
    run_ckpt_dir.mkdir(parents=True, exist_ok=True)
    run_out_dir.mkdir(parents=True, exist_ok=True)

    for fold, (train_paths, train_ages, val_paths, val_ages) in enumerate(folds):
        print(f"\n========== Fold {fold + 1}/{len(folds)} ==========")
        print(f"Train images: {len(train_paths)} | Validation images: {len(val_paths)}")

        set_seed(SEED + fold)
        train_loader, val_loader = make_loaders(
            train_paths, train_ages, val_paths, val_ages, task="classification"
        )

        model = model_builder(fold).to(DEVICE)
        criterion = nn.CrossEntropyLoss()

        if optimize_fusion_head_only:
            trainable_params = model.fusion_head.parameters()
        else:
            trainable_params = [p for p in model.parameters() if p.requires_grad]

        optimizer = AdamW(trainable_params, lr=lr, weight_decay=WEIGHT_DECAY)

        best_mae = float("inf")
        best_epoch = -1
        no_improve = 0
        history = []

        for epoch in range(1, EPOCHS + 1):
            start = time.time()
            train_loss = train_one_epoch_classification(model, train_loader, criterion, optimizer, DEVICE)
            val_metrics = evaluate_classification_as_regression(model, val_loader, criterion, DEVICE)
            elapsed = time.time() - start

            row = {
                "fold": fold,
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_mae": val_metrics["mae"],
                "val_accuracy": val_metrics["accuracy"],
                "seconds": elapsed,
            }
            history.append(row)

            print(
                f"Fold {fold} | Epoch {epoch:03d} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val MAE: {val_metrics['mae']:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f}"
            )

            if val_metrics["mae"] < best_mae:
                best_mae = val_metrics["mae"]
                best_epoch = epoch
                no_improve = 0
                save_checkpoint(
                    run_ckpt_dir / f"fold_{fold}_best.pth",
                    model,
                    optimizer,
                    fold,
                    epoch,
                    val_metrics,
                    extra_config={"lr": lr, "run_name": run_name},
                )
                print("  Saved best checkpoint.")
            else:
                no_improve += 1

            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best epoch: {best_epoch}, best MAE: {best_mae:.4f}")
                break

        hist_df = pd.DataFrame(history)
        hist_df.to_csv(run_out_dir / f"fold_{fold}_history.csv", index=False)

        best_row = hist_df.loc[hist_df["val_mae"].idxmin()].to_dict()
        all_best_rows.append(best_row)

    results_df = pd.DataFrame(all_best_rows).sort_values("val_mae").reset_index(drop=True)
    results_df.to_csv(run_out_dir / "kfold_summary.csv", index=False)

    print("\n========== K-FOLD SUMMARY ==========")
    print(results_df)
    print(f"Mean MAE: {results_df['val_mae'].mean():.4f}")
    print(f"Std MAE:  {results_df['val_mae'].std(ddof=1):.4f}")

    return results_df

In [ ]:
results_df = run_classification_kfold(build_model, RUN_NAME, LR, optimize_fusion_head_only=False)